# Chapter 19: Union-Find

The programs from Chapter 19 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Groups as Trees

`union_find.py`

In [1]:
"""Union-find (disjoint sets): which items are connected?

Each set is a tree; its root names the set. find() follows parents to
the root and then points every node on the way straight at it (path
compression). union() hangs the smaller tree under the larger (union by
size). Together they make each operation nearly constant time.
"""


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))  # each item starts as its own set
        self.size = [1] * n
        self.sets = n

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:  # compress the path
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a, b):
        """Join the sets of a and b. False if already in the same set."""
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra  # the smaller tree goes under the larger
        self.size[ra] += self.size[rb]
        self.sets -= 1
        return True

    def connected(self, a, b):
        return self.find(a) == self.find(b)


class NaiveUnionFind:
    """No compression, no sizes: trees can become long chains."""

    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while self.parent[x] != x:
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        self.parent[ra] = rb
        return True

## Friend Groups

`friend_groups.py`

In [2]:
"""Group people who are friends, directly or through others."""

from union_find import UnionFind

people = ["Ana", "Ben", "Cat", "Dan", "Eve", "Fay", "Gus"]
index = {name: i for i, name in enumerate(people)}
friendships = [
    ("Ana", "Ben"),
    ("Cat", "Dan"),
    ("Ben", "Cat"),
    ("Eve", "Fay"),
]

uf = UnionFind(len(people))
for a, b in friendships:
    uf.union(index[a], index[b])

groups = {}
for name in people:
    groups.setdefault(uf.find(index[name]), []).append(name)
print(f"{uf.sets} groups:", list(groups.values()))
print("Ana and Dan connected?", uf.connected(index["Ana"], index["Dan"]))

3 groups: [['Ana', 'Ben', 'Cat', 'Dan'], ['Eve', 'Fay'], ['Gus']]
Ana and Dan connected? True


## Timing the Tricks

`bench_union_find.py`

In [3]:
"""n unions that build one long chain, then a find on every item:
naive union-find against path compression and union by size."""

import bench
from union_find import NaiveUnionFind, UnionFind


def chain_then_find(make, n):
    uf = make(n)
    for i in range(n - 1):
        uf.union(i, i + 1)  # the naive version builds a chain
    for i in range(n):
        uf.find(i)


sizes = [1_000, 2_000, 4_000, 8_000]
results = {
    "naive": bench.doubling(
        chain_then_find, lambda n: (NaiveUnionFind, n), sizes
    ),
    "compressed": bench.doubling(
        chain_then_find, lambda n: (UnionFind, n), sizes
    ),
}
bench.table(results)
bench.chart(results, "ch19_union_find.png", "Union-find: a chain of n")

         n           naive      compressed
     1,000         10.6 ms          297 us
     2,000    43.8 ms x4.1     626 us x2.1
     4,000     175 ms x4.0    1.28 ms x2.0
     8,000     704 ms x4.0    2.64 ms x2.1
